# Inception modules & 1×1 convolutions

_Deep Learning_

**Run several filter sizes side by side, and use a cheap 1×1 conv to shrink channels first.**

A normal convolution forces you to pick one filter size. Small filters see fine detail; large filters see bigger structures. Which is right? You usually do not know.

       The Inception module sidesteps the choice: it runs several convolutions in parallel on the same input — a 1×1, a 3×3, a 5×5, and a pooling branch — then concatenates all their outputs along the channel axis. The next layer sees features at every scale and the network learns which to use.

---

This notebook is a practice scaffold for the **Inception modules & 1×1 convolutions** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We'll build a GoogLeNet-style Inception module, then run a tensor through it to see the output shape. The build comes in two parts: first define the four parallel branches, then run an input through and concatenate their outputs.

### Step 1 — Define the four parallel branches

Each branch processes the **same** input differently: a cheap 1×1 conv, a 3×3 conv (with a 1×1 reduce first to cut channels), a 5×5 conv (also reduced first), and a max-pool followed by a 1×1 projection. The `padding` values keep every branch's spatial size identical so their outputs can later be stacked. Note we only define the layers here — no data flows yet.

In [ ]:
import torch
import torch.nn as nn

class InceptionModule(nn.Module):
    """GoogLeNet-style module: 4 branches run in parallel, concatenated on channels."""
    def __init__(self, in_ch, b1, b3_reduce, b3, b5_reduce, b5, pool_proj):
        super().__init__()
        # Branch 1: a single cheap 1x1 conv
        self.branch1 = nn.Conv2d(in_ch, b1, kernel_size=1)
        # Branch 2: 1x1 reduce -> 3x3 (padding=1 keeps spatial size)
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_ch, b3_reduce, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv2d(b3_reduce, b3, kernel_size=3, padding=1),
        )
        # Branch 3: 1x1 reduce -> 5x5 (padding=2 keeps spatial size)
        self.branch5 = nn.Sequential(
            nn.Conv2d(in_ch, b5_reduce, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv2d(b5_reduce, b5, kernel_size=5, padding=2),
        )
        # Branch 4: 3x3 max-pool (same size) -> 1x1 projection
        self.branchp = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_ch, pool_proj, kernel_size=1),
        )

    def forward(self, x):
        outs = [self.branch1(x), self.branch3(x), self.branch5(x), self.branchp(x)]
        return torch.cat(outs, dim=1)   # concatenate along the channel axis (NCHW)

### Step 2 — Run an input and inspect the output

We instantiate the module with the real GoogLeNet inception(3a) channel counts and push a `(1, 192, 28, 28)` tensor through it. The forward pass runs all four branches and concatenates them, so the output channel count is the **sum** of the branches (64 + 128 + 32 + 32 = 256), while the 28×28 spatial size is preserved.

In [ ]:
# GoogLeNet inception(3a) configuration, input = 192 channels at 28x28
m = InceptionModule(192, b1=64, b3_reduce=96, b3=128,
                    b5_reduce=16, b5=32, pool_proj=32)
x = torch.randn(1, 192, 28, 28)        # (batch, channels, height, width)
y = m(x)

print("input shape :", tuple(x.shape))
print("output shape:", tuple(y.shape))                 # (1, 256, 28, 28): 64+128+32+32 = 256
print("out channels:", 64 + 128 + 32 + 32, "(sum of the 4 branches)")
print("total params:", sum(p.numel() for p in m.parameters()))   # 163,696

## Visualize the data & results

_How much compute does the 1×1 bottleneck save before an expensive 5×5 convolution?_

### Step 1 — Set up the convolution dimensions

Before comparing costs, we fix the shapes: 192 input channels, 32 output channels, a 16-channel bottleneck, a 5×5 filter, and a 28×28 output. The number of output positions is just height × width — every one of those pixels costs a full set of multiply-adds.

In [ ]:
import numpy as np

Cin, Cout, Cred = 192, 32, 16     # input / output / reduced channels
F = 5                              # the expensive filter size
H = W = 28                         # output spatial size
positions = H * W                  # number of output pixels

### Step 2 — Count the multiply-adds with and without the bottleneck

A direct 5×5 conv costs `F*F*Cin` multiply-adds per output value, times `Cout` filters, times the number of positions. The bottleneck instead does a cheap 1×1 reduce (192→16) and then runs the 5×5 on only 16 channels. Summing those two smaller costs gives the bottleneck total.

In [ ]:
# FLOPs ~= (F*F*Cin) multiply-adds per output value, times Cout filters, times positions
direct   = F * F * Cin * Cout * positions          # 120,422,400
reduce11 = 1 * 1 * Cin * Cred * positions          #   2,408,448  (1x1: 192 -> 16)
conv5    = F * F * Cred * Cout * positions          #  10,035,200  (5x5 on 16 ch)
bottleneck = reduce11 + conv5                       #  12,443,648

print("direct 5x5 :", direct)
print("1x1 reduce :", reduce11)
print("5x5 on 16  :", conv5)
print("bottleneck :", bottleneck)
print("saving     : %.2fx" % (direct / bottleneck))   # 9.68x

### Step 3 — Plot the cost comparison

The bar chart puts the four numbers side by side. The red bar is the expensive direct 5×5; the green bar is the bottleneck total — far shorter, showing the roughly 9.7× compute saving that 1×1 reductions unlock.

In [ ]:
import matplotlib.pyplot as plt
labels = ["direct 5x5", "1x1 reduce", "5x5 on 16 ch", "bottleneck total"]
values = [direct, reduce11, conv5, bottleneck]
colors = ["#ff7b72", "#9aa7b4", "#9aa7b4", "#7ee787"]
plt.bar(labels, values, color=colors)
plt.ylabel("multiply-adds (FLOPs)")
plt.title("5x5 conv: direct vs 1x1 bottleneck")
plt.xticks(rotation=15)
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A 3×3 conv goes from $C_{in}=256$ to $C_{out}=128$. How many parameters (including biases)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use $(F\times F\times C_{in}+1)\times C_{out}$ with $F=3$. — _Each of the $C_{out}$ filters has $F\times F\times C_{in}$ weights plus one bias._
- Compute $F\times F\times C_{in} = 3\times3\times256 = 2304$, add 1 to get $2305$. — _The $+1$ is the per-filter bias._
- Multiply by $C_{out}=128$: $2305\times128 = 295{,}040$. — _One filter's weight-and-bias count times the number of filters._

**Answer:** $$ (3\times3\times256+1)\times128 = 2305\times128 = 295{,}040 \text{ parameters.} $$

</details>

**Problem 2.** Insert a 1×1 bottleneck reducing $C_{in}=256$ to $C_{red}=64$ before that same 3×3 conv to $C_{out}=128$. Roughly how much do the weights drop (ignore biases)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Direct 3×3 weights: $3\times3\times256\times128 = 294{,}912$. — _This is the cost we want to beat._
- Bottleneck: 1×1 weights $= 256\times64 = 16{,}384$; 3×3 weights $= 3\times3\times64\times128 = 73{,}728$; total $= 90{,}112$. — _The big conv now runs on 64 channels instead of 256._
- Ratio $= 294{,}912 \div 90{,}112 \approx 3.27$. — _The expensive conv shrank by $256/64 = 4\times$, but the extra 1×1 cost pulls the net saving down a little._

**Answer:** About a 3.3× reduction in weights (from $\sim$295k to $\sim$90k) — close to, but less than, the $256/64 = 4\times$ channel ratio because the 1×1 layer adds its own cost.

</details>